<a href="https://colab.research.google.com/github/inojneupane/Indoor-Positioning-Industry-5.0/blob/main/11052026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

** POSITION CLASSIFICATION USING CNN IN UJIINDOORLOC WITH MIN-MAX SCALAR (REMARKS: CNN RESULTS LESS THAN RF)**

In [1]:
#### IMPORTING AND PREPARING TRAINING DATA
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from tensorflow.keras import layers, Model

## Importing training data
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/PhD/Dataset/UJIndoorLoc/trainingData.csv'
df_original = pd.read_csv(file_path) # Original training data

file_path = '/content/drive/MyDrive/PhD/Dataset/UJIndoorLoc/validationData.csv'
df_test_original = pd.read_csv(file_path)

df_Train = df_original.drop(["SPACEID", "RELATIVEPOSITION", "USERID", "TIMESTAMP", "PHONEID"], axis=1)
df_Test = df_test_original.drop(["SPACEID", "RELATIVEPOSITION", "USERID", "TIMESTAMP", "PHONEID"], axis=1)

overall_min_longitude = min(df_Train['LONGITUDE'].min(), df_Test['LONGITUDE'].min())
overall_min_latitude = min(df_Train['LATITUDE'].min(), df_Test['LATITUDE'].min())

df_Train['LONGITUDE_ADJUSTED'] = df_Train['LONGITUDE'] - overall_min_longitude
df_Train['LATITUDE_ADJUSTED'] = df_Train['LATITUDE'] - overall_min_latitude

df_Test['LONGITUDE_ADJUSTED'] = df_Test['LONGITUDE'] - overall_min_longitude
df_Test['LATITUDE_ADJUSTED'] = df_Test['LATITUDE'] - overall_min_latitude




for a in range(25, 26 ,5):

    print(f'Grid size in m x m: {a}')

    df_Train['LONGITUDE_GRID'] = (df_Train['LONGITUDE_ADJUSTED'] / a).astype(int)
    df_Train['LATITUDE_GRID'] = (df_Train['LATITUDE_ADJUSTED'] / a).astype(int)
    #df_Train['Position'] = df_Train['LONGITUDE_GRID'].astype(str) + '_' + df_Train['LATITUDE_GRID'].astype(str) + '_' + df_Train['FLOOR'].astype(str)
    df_Train['Position'] = df_Train['LONGITUDE_GRID'] * 10000 + df_Train['LATITUDE_GRID'] *100 + df_Train['FLOOR']


    df_Test['LONGITUDE_GRID'] = (df_Test['LONGITUDE_ADJUSTED'] / a).astype(int)
    df_Test['LATITUDE_GRID'] = (df_Test['LATITUDE_ADJUSTED'] / a).astype(int)
    #df_Test['Position'] = df_Test['LONGITUDE_GRID'].astype(str) + '_' + df_Test['LATITUDE_GRID'].astype(str) + '_' + df_Test['FLOOR'].astype(str)
    df_Test['Position'] = df_Test['LONGITUDE_GRID'] * 10000 + df_Test['LATITUDE_GRID'] *100 + df_Test['FLOOR']

    train_set = df_Train.copy()
    data_train = train_set.drop(["LONGITUDE", "LATITUDE", "LONGITUDE_ADJUSTED", "LATITUDE_ADJUSTED", "LONGITUDE_GRID", "LATITUDE_GRID", "Position", "FLOOR", "BUILDINGID"], axis=1) #removing target from train set
    data_train_labels_position  = train_set["Position"].copy() # placing target in label

    test_set = df_Test.copy()
    data_test = test_set.drop(["LONGITUDE", "LATITUDE", "LONGITUDE_ADJUSTED", "LATITUDE_ADJUSTED", "LONGITUDE_GRID", "LATITUDE_GRID", "Position", "FLOOR", "BUILDINGID"], axis=1) #removing target from test set
    data_test_labels_position = test_set["Position"].copy() # placing target in label

    X_train = data_train
    X_test = data_test
    y_train_position = data_train_labels_position
    y_test_position = data_test_labels_position

    unique_train_positions = df_Train['Position'].unique()
    unique_test_positions = df_Test['Position'].unique()
    #print(unique_train_positions)
    print(f"Number of unique train positions: {len(unique_train_positions)}")
    #print(unique_test_positions)
    print(f"Number of unique test positions: {len(unique_test_positions)}")

    model_position_knn = KNeighborsClassifier(n_neighbors=1)
    model_position_knn.fit(X_train, y_train_position)
    y_pred_position_knn = model_position_knn.predict(X_test)
    accuracy_knn = accuracy_score(y_test_position, y_pred_position_knn)
    f1_knn = f1_score(y_test_position, y_pred_position_knn, average='weighted')
    print(f'Position Accuracy KNN: {accuracy_knn}')
    print(f'Position F1 Score KNN: {f1_knn}')

    model_position_rf = RandomForestClassifier(random_state=42)
    model_position_rf.fit(X_train, y_train_position)
    y_pred_position_rf = model_position_rf.predict(X_test)
    accuracy_rf = accuracy_score(y_test_position, y_pred_position_rf)
    f1_rf = f1_score(y_test_position, y_pred_position_rf, average='weighted')
    print(f'Position Accuracy RF: {accuracy_rf}')
    print(f'Position F1 Score RF: {f1_rf}')

    print(f'---------------------------------------------------------------------------------------------------')


print(f'FINISHED AT GRID m x m: {a}')

Mounted at /content/drive
Grid size in m x m: 25
Number of unique train positions: 187
Number of unique test positions: 175
Position Accuracy KNN: 0.45364536453645365
Position F1 Score KNN: 0.4416048315085641
Position Accuracy RF: 0.6417641764176417
Position F1 Score RF: 0.6140749042936626
---------------------------------------------------------------------------------------------------
FINISHED AT GRID m x m: 25


In [2]:
### MIN-MAX SCALAR

WAP_min = X_train.min(axis=None) # Finding minimum WAP value in the entire dataset
X_train.replace(100, (WAP_min - 1), inplace=True) # Replacing 100 with 1 dBm less than the minimum WAP value in the entire dataset
X_train = X_train + (-1 * (WAP_min - 1))  # Making 100 to 0
WAP_max = X_train.max(axis=None) # Finding maximum WAP after adding
X_train = X_train.div(WAP_max) # Dividing by corresponding max value to normalize
X_train.head()

,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP511,WAP512,WAP513,WAP514,WAP515,WAP516,WAP517,WAP518,WAP519,WAP520
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07619,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
X_test.replace(100, (WAP_min - 1), inplace=True)
X_test = X_test + (-1 * (WAP_min - 1))
X_test = X_test.div(WAP_max)
X_test.head()

,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP511,WAP512,WAP513,WAP514,WAP515,WAP516,WAP517,WAP518,WAP519,WAP520
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
    model_position_knn = KNeighborsClassifier(n_neighbors=1)
    model_position_knn.fit(X_train, y_train_position)
    y_pred_position_knn = model_position_knn.predict(X_test)
    accuracy_knn = accuracy_score(y_test_position, y_pred_position_knn)
    f1_knn = f1_score(y_test_position, y_pred_position_knn, average='weighted')
    print(f'Position Accuracy KNN: {accuracy_knn}')
    print(f'Position F1 Score KNN: {f1_knn}')

    model_position_rf = RandomForestClassifier(random_state=42)
    model_position_rf.fit(X_train, y_train_position)
    y_pred_position_rf = model_position_rf.predict(X_test)
    accuracy_rf = accuracy_score(y_test_position, y_pred_position_rf)
    f1_rf = f1_score(y_test_position, y_pred_position_rf, average='weighted')
    print(f'Position Accuracy RF: {accuracy_rf}')
    print(f'Position F1 Score RF: {f1_rf}')

    print(f'---------------------------------------------------------------------------------------------------')




print(f'FINISHED AT GRID m x m: {a}')

Position Accuracy KNN: 0.6291629162916291
Position F1 Score KNN: 0.6133025839470938
Position Accuracy RF: 0.6921692169216922
Position F1 Score RF: 0.6649096445582358
---------------------------------------------------------------------------------------------------
FINISHED AT GRID m x m: 25


In [5]:
### Converting WAP values to images

image_width = 20 # Width of image
image_height = 26 # Height of image

num_samples = len(X_train) # Number of samples of training data
reshaped_images = np.zeros((num_samples, image_height, image_width)) # Reshaping images

for i in range(num_samples):
    pixel_values = X_train.iloc[i].values
    reshaped_images[i] = pixel_values.reshape(image_height, image_width)
X_train = reshaped_images # Putting reshaped images to X_train

In [6]:
num_samples = len(X_test) # Number of samples of testing data
reshaped_images = np.zeros((num_samples, image_height, image_width)) # Reshaping images

for i in range(num_samples):
    pixel_values = X_test.iloc[i].values
    reshaped_images[i] = pixel_values.reshape(image_height, image_width)
X_test = reshaped_images # Putting reshaped images to X_train

In [7]:
### Training position classification model

from sklearn.preprocessing import LabelEncoder

all_unique_labels = np.unique(np.concatenate((y_train_position, y_test_position)))

label_encoder = LabelEncoder()
label_encoder.fit(all_unique_labels) # Fit on all possible labels

y_train_position_encoded = label_encoder.transform(y_train_position)
y_test_position_encoded = label_encoder.transform(y_test_position)

# Update the number of units in the output layer based on the encoded labels
num_position_classes = len(label_encoder.classes_)

## Define the input layer
x = layers.Input(shape=(26, 20, 1))

## First Convolutional layer with 64 filters, 7x7 kernel, ReLU activation, and same padding
y = layers.Conv2D(64, (7, 7), activation='relu', padding="same")(x)
y = layers.MaxPooling2D(pool_size=(2, 2))(y)

## Second Convolutional layer with 128 filters, 3x3 kernel, ReLU activation, and same padding
y = layers.Conv2D(128, (3, 3), activation='relu', padding="same")(y)
y = layers.Conv2D(128, (3, 3), activation='relu', padding="same")(y)
y = layers.MaxPooling2D(pool_size=(2, 2))(y)

## Third Convolutional layer with 256 filters, 3x3 kernel, ReLU activation, and same padding
y = layers.Conv2D(256, (3, 3), activation='relu', padding="same")(y)
y = layers.Conv2D(256, (3, 3), activation='relu', padding="same")(y)
y = layers.MaxPooling2D(pool_size=(2, 2))(y)

## Flatten the output for dense layers
y = layers.Flatten()(y)

## Fully connected layers
y = layers.Dense(units=256, activation='relu')(y)
y = layers.Dropout(rate=0.5)(y)

y = layers.Dense(64, activation='relu')(y)
y = layers.Dropout(rate=0.5)(y)

# Output layer with n units (for n classes of position) and softmax activation
z = layers.Dense(units=num_position_classes, activation='softmax')(y)

model_position_cnn = Model(inputs=x, outputs=z)
model_position_cnn.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',  # For multiclass classification with integer labels
              metrics=['accuracy'])
model_position_cnn.fit(X_train, y_train_position_encoded, epochs=15, batch_size=64, validation_split=0.1)

Epoch 1/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.0237 - loss: 4.9450 - val_accuracy: 0.0928 - val_loss: 4.6723
Epoch 2/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.2033 - loss: 3.3489 - val_accuracy: 0.3531 - val_loss: 3.1072
Epoch 3/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.3839 - loss: 2.1541 - val_accuracy: 0.4769 - val_loss: 2.4374
Epoch 4/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4971 - loss: 1.6121 - val_accuracy: 0.5466 - val_loss: 2.0139
Epoch 5/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5537 - loss: 1.3427 - val_accuracy: 0.6053 - val_loss: 1.8548
Epoch 6/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6004 - loss: 1.1862 - val_accuracy: 0.6113 - val_loss: 1.9597
Epoch 7/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6261 - loss: 1.0771 - val_accuracy: 0.6093 - val_loss: 2.0255
Epoch 8/15
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6552 - loss: 0.9881 - val_accura

In [8]:
    y_pred_position_cnn = model_position_cnn.predict(X_test)
    y_pred_position_cnn_labels = np.argmax(y_pred_position_cnn, axis=1)

    accuracy_cnn = accuracy_score(y_test_position_encoded, y_pred_position_cnn_labels)
    f1_cnn = f1_score(y_test_position_encoded, y_pred_position_cnn_labels, average='weighted')
    print(f'Position Accuracy CNN: {accuracy_cnn}')
    print(f'Position F1 Score CNN: {f1_cnn}')

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
Position Accuracy CNN: 0.675967596759676
Position F1 Score CNN: 0.6492629399456821
